# Exploring the *A. oryzae* secretion benchmark

Four CSVs in `data/`, and a short tour of what you can ask them.

The tables are split the way the biology is. A study reports several
experiments; an experiment may edit several genes at once; each experiment
yields several measurements. That split keeps the records honest, but it
means nearly every real question is a join across files. Each query below
says which files it has to reach into and why.

Read-only: nothing here writes to `data/`.

In [1]:
import pandas as pd

# Read everything as text. "TODO" and "not_reported" are meaningful values in
# these tables, and letting pandas turn them into NaN would blur "nobody has
# checked yet" together with "the paper doesn't say".
def table(name):
    return pd.read_csv(f"../data/{name}.csv", dtype=str, keep_default_na=False)

studies     = table("studies")
experiments = table("experiments")
genes       = table("experiment_genes")
outcomes    = table("outcomes")

pd.DataFrame(
    [(name, len(t), t.shape[1]) for name, t in
     [("studies", studies), ("experiments", experiments),
      ("experiment_genes", genes), ("outcomes", outcomes)]],
    columns=["table", "rows", "columns"],
).set_index("table")

,rows,columns
table,,
studies,15,8
experiments,11,8
experiment_genes,13,6
outcomes,23,12


## 1. What has the field actually tried?

Which kinds of intervention show up in the literature, and which has nobody
touched?

The strategy label `gene_role` lives in the gene table, but the thing worth
counting is *experiments*, which live in another. And an experiment that edits
two genes has two gene rows, so counting rows would silently weight it double.
Hence the `drop_duplicates`.

In [2]:
(genes.drop_duplicates(["experiment_id", "gene_role"])
      .merge(experiments, on="experiment_id")
      .groupby("gene_role")
      .agg(experiments=("experiment_id", "nunique"),
           studies=("study_id", "nunique"))
      .sort_values("experiments", ascending=False))

,experiments,studies
gene_role,,
remove_protease,8,2
fix_misrouting,2,1
target_regulator,2,1


The lopsidedness is the finding. Almost everything tried so far is *delete a
protease*. The README defines eight strategy categories; most have no rows at
all, which is a fair map of where nobody has looked yet.

(An experiment using two strategies counts once under each, so the column can
total more than the number of experiments.)

## 2. Has anyone already knocked this gene out?

The reference use case. Before committing to a knockout, check whether someone
did it, what they were expressing, what they compared against, and what came
out.

No single row holds that. The gene name is in the gene table, the cargo and
control strain in the experiment table, the numbers in the outcomes table.

In [3]:
COLUMNS = ["experiment_id", "edit_notation", "cargo", "control_strain",
           "strain", "arm", "value", "unit", "vs_control", "source_ref"]

def tried(gene):
    """Every experiment that edited this gene, with its setup and results."""
    return (genes[genes.gene_name.str.lower() == gene.lower()]
            .merge(experiments, on="experiment_id")
            .merge(outcomes, on="experiment_id")
            .sort_values(["experiment_id", "arm"])[COLUMNS]
            .reset_index(drop=True))

tried("tppA")

,experiment_id,edit_notation,cargo,control_strain,strain,arm,value,unit,vs_control,source_ref
0,JIN2007_TPPA,ΔtppA::adeA,human lysozyme,NA-2L,NA-2L,control,15.6,mg/L,1.0x,Fig. 4
1,JIN2007_TPPA,ΔtppA::adeA,human lysozyme,NA-2L,NA-2L-tp5,modified,21.2,mg/L,1.36x,Fig. 4
2,JIN2007_TPPA_PALB,ΔtppA::argB,human lysozyme,N-2L,N2L-pa-tp6,modified,TODO,mg/L,TODO,Fig. 5
3,JIN2007_TPPA_PEPE,ΔtppA::argB,human lysozyme,N-2L,N-2L,control,TODO,mg/L,1.0x,Fig. 5
4,JIN2007_TPPA_PEPE,ΔtppA::argB,human lysozyme,N-2L,N2L-peE-tp6,modified,25.4,mg/L,1.63x,Fig. 5


Two experiments touch `tppA`, one alone and one paired with `pepE`. They use
*different* control strains, so their fold-changes are not two readings of the
same thing.

**One trap worth knowing.** Not every experiment has a `control` row. Where a
paper measured one control once and read several disruptants against it, that
control is stored once rather than copied, since duplicating it would record
one measurement as five. So `JIN2007_TPPA_PALB` above shows no control row. The
link that always holds is `experiments.control_strain`, which names the control
for every experiment whether or not a row sits under that `experiment_id`.

## 3. Experiments that changed more than one gene

This is the query that explains why the gene table exists at all.

A double disruption is **one experiment, two gene rows, one measurement**. You
cannot recover what each gene contributed from a strain that is missing both.
The strain has a single phenotype, and it is the joint result. A flatter schema
would force you to either invent two rows that were never separately measured,
or throw a gene away.

`JIN2007_TPPA_PALB` is the proof: it made *less* lysozyme than either single
disruption did. Two edits that each help alone hurt in combination, and nothing
in the single-gene rows predicts it.

In [4]:
multi = genes.experiment_id.value_counts().loc[lambda n: n > 1].index

combinations = (genes[genes.experiment_id.isin(multi)]
                .groupby("experiment_id")
                .agg(genes=("gene_name", ", ".join),
                     strategies=("gene_role", lambda r: ", ".join(sorted(set(r)))))
                .join(experiments.set_index("experiment_id")[["cargo", "control_strain"]]))

measurements = (outcomes[outcomes.experiment_id.isin(multi)]
                .set_index("experiment_id")
                [["strain", "arm", "value", "unit", "vs_control", "source_ref"]])

display(combinations, measurements)

,genes,strategies,cargo,control_strain
experiment_id,,,,
JIN2007_TPPA_PALB,"tppA, palB","remove_protease, target_regulator",human lysozyme,N-2L
JIN2007_TPPA_PEPE,"tppA, pepE",remove_protease,human lysozyme,N-2L


,strain,arm,value,unit,vs_control,source_ref
experiment_id,,,,,,
JIN2007_TPPA_PEPE,N-2L,control,TODO,mg/L,1.0x,Fig. 5
JIN2007_TPPA_PEPE,N2L-peE-tp6,modified,25.4,mg/L,1.63x,Fig. 5
JIN2007_TPPA_PALB,N2L-pa-tp6,modified,TODO,mg/L,TODO,Fig. 5


## 4. How big are the effects, by strategy?

Another join, for two reasons. `vs_control` is stored as text (`2.9x`), because
that is how it reads in the source, so it needs parsing before it can be
averaged. And the strategy label sits in a different file again.

**These numbers do not compare across studies.** Every study used its own
construct, control strain, and culture conditions; `JIN2007` alone expresses two
tandem copies of the cargo where the others express one, putting its milligram
figures on a different scale entirely. A fold-change here means "beat *its own*
control by this much", and nothing more. Read the table to spot something worth
a closer look, not to rank interventions.

In [5]:
def fold(value):
    """'2.9x' -> 2.9, 'TODO' -> missing."""
    try:
        return float(value.rstrip("x"))
    except ValueError:
        return float("nan")

# A multi-gene experiment joins to one row per gene, so it appears under each
# strategy it used; drop_duplicates stops it counting twice within one strategy.
scored = (outcomes.query("arm == 'modified'")
          .assign(fold=lambda d: d.vs_control.map(fold))
          .merge(genes[["experiment_id", "gene_role"]], on="experiment_id")
          .drop_duplicates(["outcome_id", "gene_role"]))

scored.groupby("gene_role").agg(
    measurements=("outcome_id", "size"),
    with_a_number=("fold", "count"),
    median=("fold", "median"),
    best=("fold", "max"),
).sort_values("median", ascending=False).round(2)

,measurements,with_a_number,median,best
gene_role,,,,
fix_misrouting,4,4,2.35,3.00
remove_protease,12,8,1.66,2.90
target_regulator,2,1,1.14,1.14


`with_a_number` is lower than `measurements` wherever values are still `TODO`,
so the medians are computed only over what has been transcribed so far.

## What this dataset can't tell you yet

The queries run. The honest summary is that the dataset is still too small for
most of what you would want to ask it.

- **It can't rank strategies.** Nearly every curated experiment deletes a
  protease. Any apparent ordering between categories says which papers happen
  to be curated, not which strategy works better.
- **It can't compare across studies.** Different constructs, cargo copy
  numbers, controls, and conditions, with no shared baseline to normalise
  against.
- **It can't predict combinations.** `JIN2007_TPPA_PALB` already breaks
  additivity. One counterexample rules out the easy assumption without
  supplying anything to replace it.
- **It can't separate cargo from host.** Chymosin and lysozyme respond
  differently to the same edit, and there are too few cargo proteins here to
  tell a cargo-specific effect from a general one.
- **It isn't fully transcribed.** Values still marked `TODO` sit in figures
  nobody has digitised. The ones already filled in are the ones stated in paper
  text, so the subset carrying numbers is not a random sample.

What it does do well is the negative space: it shows which strategies nobody
has tried, and it keeps the combinations and the failures that a summary of
headline results would quietly drop.